# 36 - Reading session 08: why we still dither every frame

**Purpose.** Explain what notebook `35` found, for someone deciding how often to dither. `35` is
written for someone *checking* it; this one is slow on purpose.

It explains four things: what a dither buys; what the night measured about dead time; what stacking
lost when four frames shared a position; and why a result this close still comes out as "drop it".

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `cadence_constants.json` and `cadence_frames.csv`, with session 06's `stack_constants`
and `sky_constants` beside them. If anything here disagrees with `results/`, `results/` is right
and this notebook is the bug. Not a second look at the gain: that is `34`, and this night only
closes the question it left open.

**It assumes** `34_dither_flip_read.ipynb`, which explains why dead time is the lever.

**The headline, recomputed below:** dithering every 4th frame saves 25 s of dead time per sub, worth
up to 9.5% SNR at 120 s. Four frames at one position let the sensor's fixed pattern line up, and
small stacks lose about 10% to it. The two roughly cancel. The rule written before the night asked
for a clear win at every stack size, and this is not one. **Keep dithering every frame, keep gain 50.**

In [ ]:
import json
import pathlib
import sys

import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

RESULTS = pathlib.Path.cwd().parent / "results"
cc = M.Constants.load(RESULTS / "cadence_constants.json")
frames = pd.read_csv(RESULTS / "cadence_frames.csv")
stack06 = json.loads((RESULTS / "stack_constants.json").read_text())
sky06 = json.loads((RESULTS / "sky_constants.json").read_text())


def v(name):
    return cc.entry(name)["value"]


def u(name):
    return cc.entry(name)["uncertainty"]


print(v("frames"))
print(v("cadence_decision"))

## 1. What a dither buys

Some pixels always read a little high and some a little low. That pattern is fixed to the
**sensor**, not to the sky. If the sky lands on new pixels every frame, the pattern lands somewhere
new every frame too, and stacking averages it down like any other noise. If four frames sit at one
position, their patterns sit on top of each other: for those four, it adds up instead of averaging.

How big is it? `35` measured it directly: frame pairs at **different** positions against pairs at
the **same** one. The difference between the two is the part locked to the sensor.

In [ ]:
share = pd.DataFrame(v("fixed_pattern_share")).T
share.columns = [f"bin {c}" for c in share.columns]
print("share of a registered single frame's noise fixed to the sensor")
print(share.round(2).to_string())

**At the 4x4 binning the ladder uses, about 40% of the noise in the greens is fixed pattern.** It is
more after binning because binning averages the random part down and leaves the fixed part alone.
That is the thing dithering exists to break up, and it is large.

## 2. What the night measured about dead time

Session 06 dithered every frame, so it only saw download and settle together. Tonight three frames
in four had no dither in front of them, so the gaps split cleanly into two kinds.

In [ ]:
gaps = frames[~frames.interruption].dropna(subset=["gap_s"])
print(gaps.groupby("dither").gap_s.describe()[["count", "mean", "50%", "min", "max"]].round(2))
print()
for name in ("download_s", "dither_settle_s", "t_dead", "t_dead_rule4", "t_dead_every_frame"):
    print(f"{name:20s} {v(name):7.2f} +/- {u(name):.2f} s")
print()
print("protocol predicted: download 0.68 s, settle 31-35 s, t_dead at every 4th 8.4-9.4 s")
print(f"session 06, every frame: {sky06['t_dead']['value']:.1f} +/- {sky06['t_dead']['uncertainty']:.1f} s")

**Every prediction landed.** The download is the archive's 0.68 s, now measured on our own frames.
The settle is about 34 s. So:

- dithering **every frame** costs about **35 s** per sub;
- dithering **every 4th** costs about **9 s** per sub.

At 120 s subs that is 155 s against 129 s per frame. Same night, about 20% more frames.

**Why `t_dead` is 9.23 and not 8.64.** Both are published. The flip and two autofocus runs were
cut out as interruptions, and each one also held a dither. So the counted gaps hold slightly fewer
dithers than one in four, and the plain night average comes out a little low. `download + settle / 4`
is what a clean night at this cadence costs. The two agree within their errors, which is also the
check that the formula works.

## 3. What stacking lost

`eta_comb` is how close a stack gets to the ideal `sqrt(N)` noise drop. 1.0 is perfect. The decision
quantity is tonight's `eta_comb` (4 frames per position) over session 06's (every frame), at the
same stack size.

In [ ]:
ratio = pd.DataFrame({"ratio": v("eta_ratio_vs_every_frame"), "+/-": u("eta_ratio_vs_every_frame")})
ratio["tonight, 4 per position"] = pd.Series(v("eta_comb_cadence"))
ratio["session 06, every frame"] = pd.Series(stack06["eta_comb_registered"]["value"])
ratio.index = ratio.index.astype(int)
dec = v("cadence_decision")
ratio["passes"] = ratio.ratio >= dec["threshold"]
print(ratio.sort_index().round(3).to_string())
print(f"\nthreshold {dec['threshold']} (download came out {dec['download_regime']})")
print(f"break-even from tonight's own dead times {dec['break_even_measured']:.3f}")

**Small stacks fail, larger ones pass.** At N = 3 and 4 every frame in the stack is from **one**
position, so the fixed pattern is not averaged at all. At N = 16, four positions share the load.

**Read the right-hand columns too.** Part of why the ratio climbs is that session 06's own
every-frame ladder starts to fall at 16 and 18. That is the only every-frame ladder there is,
so the ratio above 8 says less than it seems.

**Is the loss real, or is it two different nights?** Tonight also holds a control: one frame from
each position, which is a stack dithered every frame, from the same night and the same setpoint.

In [ ]:
ctrl = pd.DataFrame({"4 per position": v("eta_comb_cadence"), "1 per position": v("eta_comb_one_per_position")})
ctrl.index = ctrl.index.astype(int)
ctrl = ctrl.dropna().sort_index()
ctrl["same-night ratio"] = ctrl["4 per position"] / ctrl["1 per position"]
print(ctrl.round(3).to_string())

**The same shape, without a second night.** 0.92 and 0.91 at 3 and 4, 0.97 at 8. The loss is the
cadence, not the change of night or setpoint.

## 4. Why a result this close comes out as "drop it"

Put the two halves side by side. Dithering every 4th frame:

- **gains** about 9.5% SNR from dead time, *if stacking lost nothing*;
- **loses** about 10% to stacking at the small rungs, and 3-7% at the middle ones.

Break-even from tonight's own numbers is a ratio of **0.914**. The two failing rungs, 0.898 and 0.902,
are under it by less than one standard error. So this is not a loss you can prove. It is not a win
either.

The rule was fixed before the night: keep the change only if it passes at **every** matched rung. It
was written that way so the night could not talk us into a change that only maybe helps. It does
exactly that job here. A cadence change that helps on some stack sizes, hurts on others and on
balance comes out even is not worth a different habit at the mount.

**What would have to be true for it to win.** Stacks only ever large, and a fixed pattern that keeps
averaging down with more positions. The rungs past 18 hint the other way: tonight reads 0.90 at 32
and at 56, with no every-frame ladder that long to compare against.

In [ ]:
print({n: round(e, 3) for n, e in v("eta_comb_cadence").items()})

## 5. The by-products

**Star clipping on a second night.** Same field, same setting, the same stars on both nights.

In [ ]:
c = v("clipped_star_fraction")
print(f"tonight   {100 * c['tonight']['clipped']['any']:.2f}% clipped in some plane")
print(f"session 06 cell B   {100 * c['session06_B']['clipped']['any']:.2f}%")
print("tonight by focus position:", {k: f"{100 * x:.2f}%" for k, x in v("clipped_star_fraction_by_focus").items()})

**About a sixth more stars clipped than last time.** The spread inside tonight, across three focus
positions, is a third of that. So one night's curve is good to roughly 15-20% of its value on
another night. That is fine for picking gain 50 at about 1% clipped; it is not fine for splitting hairs
between 1.0% and 1.2%.

**The sky.** `F_sky` green came out 2.13 e-/px/s against session 06's 1.93: about 10% brighter.
Recorded, not acted on.

## 6. What is settled, and what is next

**Settled.**
- **Dither every frame.** Every 4th frame saves real time and loses about as much to stacking. Under
  the rule fixed beforehand, that is a no.
- **The dead time is measured.** Download 0.69 s, settle 34 s. Nothing about it is inferred any more.
- **Gain 50 stays.** Gain 200 needed dithering every 12th frame or rarer. Every 4th already gives its
  gain back, so the flip is out of practical reach.

**Not settled, named rather than fixed.**
1. **`eta_comb` past 18 in the model.** The model still uses the bias stack there (0.79 at 32), which
   its note calls an upper bound on the loss. Tonight's on-sky ladder reads 0.90 at 32, but at the
   wrong cadence. Whether the model should take on-sky numbers is a conversation.
2. **Session 06's every-frame `t_dead`, 31.5 +/- 10.4 s**, is wide. Tonight's every-frame
   equivalent, 34.9 +/- 1.9 s, sits inside it and is much tighter. Nothing moves until that is
   decided.